[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Seguridad de dependencias e imagenes.

Este cuaderno orienta la seguridad a dos frentes de Py271: dependencias del proyecto y superficie de ataque de las imagenes publicadas por el pipeline.

## Objetivos.

- Reforzar buenas practicas de hardening en Actions.
- Relacionar severidad con politicas de bloqueo.
- Introducir el control de actions y dependencias de terceros.

## Recomendaciones base.

Retomamos varias practicas de endurecimiento:

- no guardar secretos en texto plano;
- vigilar cambios a workflows con CODEOWNERS;
- desconfiar de entradas no fiables e inyecciones de script;
- revisar actions de terceros y preferir versiones fijadas;
- usar Dependabot para mantener referencias actualizadas.

Una politica de severidad util distingue entre hallazgos que bloquean automaticamente y hallazgos que requieren analisis compensatorio.

## SCA: analisis de composicion de software.

El analisis de composicion de software (SCA, por sus siglas en ingles) consiste en identificar todos los componentes de terceros que forman parte de una aplicacion y verificar si alguno tiene vulnerabilidades conocidas (CVEs).

En una aplicacion Python moderna, esos componentes son tres capas:

```text
Imagen de contenedor
  ├── Sistema operativo base (ej. python:3.12-slim = Debian)
  │     └── paquetes del OS: openssl, libexpat, zlib, ...
  └── Aplicacion
        └── dependencias Python: requests, apiflask, gunicorn, ...
```

Cualquiera de esas capas puede contener un CVE publicado. La imagen base que funcionaba hace tres meses puede tener hoy vulnerabilidades criticas en sus paquetes de sistema, aunque el codigo de la aplicacion no haya cambiado.

SCA en el pipeline responde a dos preguntas concretas:
- que version de cada componente estamos usando;
- si alguna version tiene un CVE conocido con severidad relevante.

## Trivy: escaner unificado de vulnerabilidades.

`Trivy` es un escaner de seguridad de codigo abierto desarrollado por Aqua Security. En un solo binario cubre los dos frentes de este capitulo: dependencias del proyecto e imagenes de contenedor.

Lo que Trivy puede escanear:

| Objetivo | Que detecta |
|----------|-------------|
| Imagen de contenedor | CVEs en paquetes del OS (apt, apk) y en dependencias de lenguaje |
| Sistema de archivos / lockfile | CVEs en paquetes Python, Node, Go, etc. declarados en el proyecto |
| Repositorio git | Secretos expuestos, configuraciones inseguras, IaC mal configurada |
| Dockerfile | Malas practicas de construccion (tambien cubre parte de lo que hace Hadolint) |

Trivy consulta multiples bases de datos de vulnerabilidades: NVD (NIST), GitHub Advisory Database, y bases especificas por distribucion (Alpine, Debian, RHEL). Las actualiza automaticamente antes de cada escaneo.

La diferencia con Dependabot es el alcance: Dependabot actualiza versiones; Trivy informa sobre vulnerabilidades conocidas en las versiones que ya tienes, incluyendo las del sistema operativo base de la imagen.

## Trivy: comandos principales.

Trivy opera en varios modos segun el objetivo del escaneo:

```bash
# escanear una imagen de contenedor
trivy image mi-app:latest

# escanear solo vulnerabilidades CRITICAL y HIGH, salir con error si hay hallazgos
trivy image --severity CRITICAL,HIGH --exit-code 1 mi-app:latest

# escanear el sistema de archivos del proyecto (dependencias, configs, secretos)
trivy fs .

# escanear el lockfile directamente
trivy fs --scanners vuln uv.lock

# generar reporte en formato SARIF (compatible con GitHub Security)
trivy image --format sarif --output resultado.sarif mi-app:latest

# escanear un repositorio remoto
trivy repo https://github.com/org/repo
```

El flag `--exit-code 1` es el que convierte a Trivy en un quality gate real: el comando retorna codigo de salida 1 si hay hallazgos con la severidad especificada, lo que hace fallar el paso del workflow y detiene el pipeline.

## Trivy en GitHub Actions.

La integracion recomendada usa la Action oficial de Aqua Security. El job escanea la imagen recien construida y falla si encuentra vulnerabilidades de severidad CRITICAL o HIGH:

```yaml
jobs:
  scan:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Construir imagen
        run: docker build -t mi-app:${{ github.sha }} .

      - name: Escanear imagen con Trivy
        uses: aquasecurity/trivy-action@0.20.0
        with:
          image-ref: mi-app:${{ github.sha }}
          format: table
          exit-code: 1
          severity: CRITICAL,HIGH
          ignore-unfixed: true

      - name: Escanear dependencias del proyecto
        uses: aquasecurity/trivy-action@0.20.0
        with:
          scan-type: fs
          scan-ref: .
          format: sarif
          output: trivy-results.sarif

      - name: Subir resultados a GitHub Security
        uses: github/codeql-action/upload-sarif@v3
        with:
          sarif_file: trivy-results.sarif
```

El formato `sarif` permite ver los hallazgos directamente en la pestana **Security > Code scanning** del repositorio en GitHub, sin salir de la plataforma.

## Dependabot: actualizaciones automaticas de dependencias.

Dependabot es una funcionalidad de GitHub que abre pull requests automaticamente cuando detecta versiones mas recientes de las dependencias declaradas en el proyecto. Se configura mediante el archivo `.github/dependabot.yml`:

```yaml
version: 2
updates:
  - package-ecosystem: "pip"
    directory: "/"
    schedule:
      interval: "weekly"
    open-pull-requests-limit: 5
    groups:
      dev-dependencies:
        patterns: ["pytest*", "ruff", "mypy"]

  - package-ecosystem: "github-actions"
    directory: "/"
    schedule:
      interval: "weekly"
```

El bloque `github-actions` es importante en un flujo DevSecOps: mantiene las Actions de terceros en versiones recientes, reduciendo la exposicion a vulnerabilidades en dependencias del pipeline mismo.

Cada PR de Dependabot pasa por el mismo pipeline de CI que cualquier otro cambio, garantizando que la actualizacion no rompa el proyecto antes de hacer merge.

## uv y la cadena de seguridad de dependencias.

`uv` contribuye a la seguridad de dependencias de dos formas complementarias:

**Lockfile con hashes verificados.** `uv.lock` incluye el hash SHA-256 de cada paquete descargado. Al instalar con `uv sync --frozen`, uv verifica que cada archivo descargado coincida exactamente con el hash registrado. Esto hace imposible un ataque de sustitucion silenciosa de paquetes.

**Reproducibilidad auditada.** Dado que `uv.lock` es un archivo de texto versionado en git, cualquier cambio en las dependencias produce un diff visible en el pull request. El equipo puede revisar que se agrego o actualizo antes de aprobar el merge.

```text
pyproject.toml   <- intenciones: "requests >= 2.31"
uv.lock          <- realidad fijada: requests==2.32.3 sha256=abc123...
CI               <- uv sync --frozen verifica hashes antes de instalar
```

Trivy puede escanear directamente el `uv.lock` para detectar CVEs en las versiones fijadas, cerrando el ciclo entre gestion de dependencias y seguridad.

## Politica de severidad.

No todos los hallazgos tienen el mismo peso. Una politica util clasifica los resultados por severidad y define que nivel de hallazgo detiene automaticamente el pipeline:

| Severidad | Accion recomendada |
|-----------|-------------------|
| `CRITICAL` | Bloqueo inmediato. El pipeline no avanza. |
| `HIGH`     | Bloqueo inmediato. Requiere remediacion antes del merge. |
| `MEDIUM`   | Warning. Se registra pero no bloquea; debe resolverse en el sprint. |
| `LOW`      | Informativo. Se reporta para seguimiento. |

La celda siguiente simula esta logica de decision en Python.

In [ ]:
vulnerabilidades = {'critical': 0, 'high': 1, 'medium': 3}
bloquea = vulnerabilidades['critical'] > 0 or vulnerabilidades['high'] > 0

print('hallazgos:', vulnerabilidades)
print('bloquear pipeline:', bloquea)

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>